# Worker B: Knapsack alpha=0.5 (Full Scale)

Runs the alpha-fair knapsack experiment at alpha=0.5 across all unfairness levels.

**Grid:** 7 methods x 5 seeds x 3 unfairness levels = 105 runs

Checkpointed: re-run cells to resume from where you left off.

In [1]:
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

try:
    import mosek
    print(f'MOSEK installed: {mosek.Env().getversion()}')
except ImportError:
    print('Installing MOSEK...')
    os.system('pip install mosek -q')
    print('MOSEK installed.')

for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}

# Patch solver chain: MOSEK > CLARABEL > SCS
import fair_dfl.tasks.md_knapsack as _kn
import cvxpy as cp
_kn._SOLVER_CHAIN = [
    (cp.MOSEK,    {}),
    (cp.CLARABEL, {}),
    (cp.SCS,      {'eps': 1e-6, 'max_iters': 10000}),
]

# Results directory
KN_RESULTS = os.path.join(DRIVE_ROOT, 'results', 'final', 'knapsack')
os.makedirs(KN_RESULTS, exist_ok=True)

from experiments.colab_runner import *
print(f'Ready. Results: {KN_RESULTS}')

Mounted at /content/drive
MOSEK license set: /content/drive/MyDrive/DecisionFocusedMTL/data/mosek.lic
Installing MOSEK...
MOSEK installed.
/content/drive/MyDrive/DecisionFocusedMTL
Ready. Results: /content/drive/MyDrive/DecisionFocusedMTL/results/final/knapsack


## Configuration

In [2]:
# ===== TASK PARAMETERS =====
TASK_OVERRIDES = {
    'n_items': 7,
    'n_budget_dims': 1,            # 1 = classic knapsack
    'n_features': 5,
    'budget_tightness': 0.3,       # tighter budget = more competition
    'poly_degree': 2,
    'decision_mode': 'group',
    'n_samples_train': 400,
    'n_samples_val': 80,           # validation split
    'n_samples_test': 200,
}

# ===== UNFAIRNESS LEVELS =====
# SNR = group_bias / noise_std_hi > 1.5 for learnable signal
UF_CONFIGS = {
    'mild':   {'group_bias': 0.2, 'noise_std_lo': 0.05, 'noise_std_hi': 0.10, 'group_ratio': 0.5},
    'medium': {'group_bias': 0.4, 'noise_std_lo': 0.05, 'noise_std_hi': 0.20, 'group_ratio': 0.6},
    'high':   {'group_bias': 0.6, 'noise_std_lo': 0.05, 'noise_std_hi': 0.30, 'group_ratio': 0.75},
}

# ===== TRAINING PARAMETERS =====
TRAIN_OVERRIDES = {
    'optimizer': 'adamw',
    'lr': 0.001,
    'weight_decay': 1e-4,
    'lr_warmup_steps': 5,
    'decision_grad_backend': 'spsa',
    'batch_size': 32,              # same for all methods
    'init_mode': 'best_practice',
    'dropout': 0.1,
    'hidden_dim': 64,
    'n_layers': 2,
}

# ===== EXPERIMENT CONTROL =====
STEPS = 70
OVERWRITE = False

print('='*60)
print('KNAPSACK alpha=0.5 — FULL SCALE')
print('='*60)
print(f'Steps: {STEPS}, Overwrite: {OVERWRITE}')
print(f'Task: {TASK_OVERRIDES}')
for k, v in UF_CONFIGS.items():
    print(f'  {k}: bias={v["group_bias"]}, noise_hi={v["noise_std_hi"]}, SNR~{v["group_bias"]/v["noise_std_hi"]:.1f}')
print(f'Train: {TRAIN_OVERRIDES}')
print('='*60)

KNAPSACK alpha=0.5 — FULL SCALE
Steps: 70, Overwrite: False
Task: {'n_items': 7, 'n_budget_dims': 1, 'n_features': 5, 'budget_tightness': 0.3, 'poly_degree': 2, 'decision_mode': 'group', 'n_samples_train': 400, 'n_samples_val': 80, 'n_samples_test': 200}
  mild: bias=0.2, noise_hi=0.1, SNR~2.0
  medium: bias=0.4, noise_hi=0.2, SNR~2.0
  high: bias=0.6, noise_hi=0.3, SNR~2.0
Train: {'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.0001, 'lr_warmup_steps': 5, 'decision_grad_backend': 'spsa', 'batch_size': 32, 'init_mode': 'best_practice', 'dropout': 0.1, 'hidden_dim': 64, 'n_layers': 2}


## Run All Unfairness Levels

In [3]:
results = run_knapsack_slice(
    alphas=[0.5],
    unfairness_levels=list(UF_CONFIGS.keys()),
    results_dir=KN_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    unfairness_configs=UF_CONFIGS,
    train_overrides=TRAIN_OVERRIDES,
    overwrite=OVERWRITE,
)


Knapsack slice: ['FPTO', 'SAA', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']
Alphas=[0.5], Unfairness=['mild', 'medium', 'high']
Seeds: [11, 22, 33, 44, 55]
Task overrides: {'n_items': 7, 'n_budget_dims': 1, 'n_features': 5, 'budget_tightness': 0.3, 'poly_degree': 2, 'decision_mode': 'group', 'n_samples_train': 400, 'n_samples_val': 80, 'n_samples_test': 200}
Train overrides: {'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.0001, 'lr_warmup_steps': 5, 'decision_grad_backend': 'spsa', 'batch_size': 32, 'init_mode': 'best_practice', 'dropout': 0.1, 'hidden_dim': 64, 'n_layers': 2}
  mild: {'group_bias': 0.2, 'noise_std_lo': 0.05, 'noise_std_hi': 0.1, 'group_ratio': 0.5}
  medium: {'group_bias': 0.4, 'noise_std_lo': 0.05, 'noise_std_hi': 0.2, 'group_ratio': 0.6}
  high: {'group_bias': 0.6, 'noise_std_lo': 0.05, 'noise_std_hi': 0.3, 'group_ratio': 0.75}
Total runs=105
  [1/105] FPTO a=0.5 uf=mild s=11 (15.4s)
    Est. remaining: 27min
  [2/105] FPTO a=0.5 uf=mild s

## Results

In [4]:
show_progress(KN_RESULTS, 'Knapsack alpha=0.5 — ALL LEVELS')

if not results.empty:
    cols = ['test_regret_normalized', 'test_fairness', 'test_pred_mse']
    for uf in sorted(results['unfairness_level'].unique()):
        sub = results[results['unfairness_level'] == uf]
        print(f'\n{"="*60}')
        print(f'alpha=0.5, unfairness={uf} (mean +/- std over seeds)')
        print(f'{"="*60}')
        summary = sub.groupby(['method_label', 'lambda'])[cols].agg(['mean', 'std']).round(4)
        print(summary.to_string())

        saa = sub[sub['method_label'] == 'SAA']
        if not saa.empty:
            saa_reg = saa['test_regret_normalized'].mean()
            print(f'\nSAA regret: {saa_reg:.4f}')
            for m in ['FPTO', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']:
                ms = sub[(sub['method_label'] == m) & (sub['lambda'] == 0.0)]
                if not ms.empty:
                    r = ms['test_regret_normalized'].mean()
                    print(f'  {m:15s}: {r:.4f} ({(r-saa_reg)/saa_reg*100:+.1f}% vs SAA)')


Knapsack alpha=0.5 — ALL LEVELS — 390 rows from 210 runs
Methods: ['FDFL-CAGrad', 'FDFL-MGDA', 'FDFL-PCGrad', 'FDFL-Scal', 'FPTO', 'SAA', 'WDRO']
Seeds:   [np.int64(11), np.int64(22), np.int64(33), np.int64(44), np.int64(55)]
Alphas:  [np.float64(0.5), np.float64(2.0)]

--- alpha = 0.5 (mean +/- std over seeds) ---
                     test_regret_normalized_mean  test_regret_normalized_std  test_fairness_mean  test_fairness_std  test_pred_mse_mean  test_pred_mse_std
method_label lambda                                                                                                                                       
FDFL-CAGrad  0.0000                       0.0115                      0.0005              0.0751             0.0474              0.7699             0.0698
FDFL-MGDA    0.0000                       0.0124                      0.0009              0.0897             0.0697              0.9377             0.1136
FDFL-PCGrad  0.0000                       0.0121              

Worker B complete. Run the Results notebook to analyze.